# Problems 6 and 7

This notebook keeps Problem 6 simple while using the same MNL model specification from Problem 1: hotel utility is a hotel intercept plus the eight assigned hotel features, with the no-purchase option normalized to utility 0.

## Problem 6 Plan

Problem 6 asks us to define customer types differently from Problem 4, estimate a new mixture MNL, and repeat the Problem 5 assortment comparison.

We will use **Saturday-night booking vs. non-Saturday booking** because the PDF explicitly suggests this variable.

- Type 1: `srch_saturday_night_bool == 1`
- Type 2: `srch_saturday_night_bool == 0`

Implementation outline:

1. Load `data/data.csv` for estimation.
2. Add a no-purchase outside option to each search query.
3. Estimate one Problem 1-style MNL model for each type using the same likelihood structure from Problem 1.
4. Use `data1.csv` through `data4.csv` for the assortment comparison.
5. Compare `S`, `S1`, and `S2` exactly like Problem 5.

In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from ortools.linear_solver import pywraplp


In [37]:
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "data.csv").exists()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"
FULL_DATA = DATA_DIR / "data.csv"
ASSORTMENT_FILES = [DATA_DIR / f"data{i}.csv" for i in range(1, 5)]


FEATURES = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag",
]

TYPE_COL = "problem6_type"
TYPE_LABELS = {1: "saturday_night", 2: "non_saturday_night"}
PARAM_NAMES = ["intercept"] + FEATURES
PRICE_COL = "price_usd"

## Load And Segment The Data

Problem 6 uses the full choice-estimation data, `data/data.csv`, then repeats the assortment comparison on `data1.csv` through `data4.csv`.

In [38]:
df = pd.read_csv(FULL_DATA).dropna(subset=["srch_id", "booking_bool", "srch_saturday_night_bool", *FEATURES])
df[TYPE_COL] = np.where(df["srch_saturday_night_bool"].astype(int).eq(1), 1, 2)

query_types = df[["srch_id", TYPE_COL]].drop_duplicates()
theta = query_types[TYPE_COL].value_counts(normalize=True).sort_index()

theta

problem6_type
1   0.576610
2   0.423390
Name: proportion, dtype: float64

## Estimate The Two Problem 1 MNL Models

This reuses the same MNL likelihood from Problem 1: hotel utility is an intercept plus the eight hotel features, and the no-purchase outside option has utility 0.

In [39]:
def compute_utilities(beta, X):
    return beta[0] + X @ beta[1:]


def neg_log_likelihood(beta, X, y, groups):
    u = compute_utilities(beta, X)
    log_likelihood = 0.0

    for q in np.unique(groups):
        mask = groups == q
        u_q = u[mask]
        y_q = y[mask]

        log_likelihood += y_q @ u_q
        log_likelihood -= logsumexp(np.append(u_q, 0.0))

    return -log_likelihood


df_saturday = df[df[TYPE_COL] == 1]
X_saturday = df_saturday[FEATURES].to_numpy(dtype=float)
y_saturday = df_saturday["booking_bool"].to_numpy(dtype=float)
groups_saturday = df_saturday["srch_id"].to_numpy()

result_saturday = minimize(
    neg_log_likelihood,
    np.zeros(len(PARAM_NAMES)),
    args=(X_saturday, y_saturday, groups_saturday),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)
beta_saturday = result_saturday.x

df_non_saturday = df[df[TYPE_COL] == 2]
X_non_saturday = df_non_saturday[FEATURES].to_numpy(dtype=float)
y_non_saturday = df_non_saturday["booking_bool"].to_numpy(dtype=float)
groups_non_saturday = df_non_saturday["srch_id"].to_numpy()

result_non_saturday = minimize(
    neg_log_likelihood,
    np.zeros(len(PARAM_NAMES)),
    args=(X_non_saturday, y_non_saturday, groups_non_saturday),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)
beta_non_saturday = result_non_saturday.x

In [40]:
beta_table = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_saturday_night": beta_saturday,
    "beta_non_saturday_night": beta_non_saturday,
})
beta_table["difference"] = beta_table["beta_saturday_night"] - beta_table["beta_non_saturday_night"]
beta_table

,parameter,beta_saturday_night,beta_non_saturday_night,difference
0,intercept,-2.780525,-2.806086,0.025561
1,prop_starrating,0.419705,0.548074,-0.128369
2,prop_review_score,0.117315,0.118357,-0.001042
3,prop_brand_bool,0.225150,0.234683,-0.009533
4,prop_location_score,0.016804,0.009699,0.007104
5,prop_accesibility_score,0.434962,0.622467,-0.187505
6,prop_log_historical_price,-0.035196,-0.042495,0.007299
7,price_usd,-0.006349,-0.008587,0.002239
8,promotion_flag,0.481328,0.418200,0.063128


## Repeat Problem 5

This follows the same structure as the actual Problem 5 notebook: use an MILP for the unknown-type mixture assortment `S`, and use the revenue-ordered greedy rule for the known single-type assortments `S1` and `S2`.

In [41]:
def compute_preference_weights(df_hotels, beta):
    u = compute_utilities(beta, df_hotels[FEATURES].to_numpy(dtype=float))
    return np.exp(np.clip(u, -50, 50))


def expected_revenue_type(x, revenue, v):
    return np.sum(x * revenue * v) / (1 + np.sum(x * v))


def optimal_assortment_single_type(df_hotels, v, price_col=PRICE_COL):
    df_sorted = df_hotels.copy()
    df_sorted["_original_index"] = np.arange(len(df_sorted))
    df_sorted["_v"] = v
    df_sorted = df_sorted.sort_values(price_col, ascending=False).reset_index(drop=True)

    numerator = 0.0
    denominator = 1.0
    chosen_original_indices = []

    for _, row in df_sorted.iterrows():
        current_revenue = numerator / denominator
        if row[price_col] > current_revenue:
            numerator += row[price_col] * row["_v"]
            denominator += row["_v"]
            chosen_original_indices.append(int(row["_original_index"]))
        else:
            break

    selected = np.zeros(len(df_hotels), dtype=int)
    selected[chosen_original_indices] = 1
    return selected, numerator / denominator


def solve_assortment_ortools(revenue, v_by_type, theta_by_type, time_limit_seconds=300):
    revenue = np.asarray(revenue, dtype=float)
    v_by_type = [np.asarray(v, dtype=float) for v in v_by_type]
    n = len(revenue)
    K = len(v_by_type)

    solver = pywraplp.Solver.CreateSolver("CBC")
    solver.SetTimeLimit(int(time_limit_seconds * 1000))

    hotels = range(n)
    types = range(K)

    x = {j: solver.BoolVar(f"x_{j}") for j in hotels}
    z = {k: solver.NumVar(0.0, 1.0, f"z_{k}") for k in types}
    y = {(j, k): solver.NumVar(0.0, 1.0, f"y_{j}_{k}") for j in hotels for k in types}

    for k in types:
        solver.Add(z[k] + solver.Sum(v_by_type[k][j] * y[j, k] for j in hotels) == 1)

    for j in hotels:
        for k in types:
            solver.Add(y[j, k] <= z[k])
            solver.Add(y[j, k] <= x[j])
            solver.Add(y[j, k] >= z[k] - (1 - x[j]))

    objective = solver.Sum(
        theta_by_type[k] * revenue[j] * v_by_type[k][j] * y[j, k]
        for k in types
        for j in hotels
    )

    solver.Maximize(objective)
    status = solver.Solve()

    selected = np.array([int(round(x[j].solution_value())) for j in hotels])
    return selected, solver.Objective().Value(), status


def interpret_status(status):
    if status == pywraplp.Solver.OPTIMAL:
        return "OPTIMAL"
    elif status == pywraplp.Solver.FEASIBLE:
        return "FEASIBLE"
    elif status == pywraplp.Solver.INFEASIBLE:
        return "INFEASIBLE"
    elif status == pywraplp.Solver.UNBOUNDED:
        return "UNBOUNDED"
    else:
        return f"OTHER ({status})"

In [42]:
rows = []

for path in ASSORTMENT_FILES:
    df_hotels = pd.read_csv(path)
    revenue = df_hotels[PRICE_COL].to_numpy(dtype=float)

    v1 = compute_preference_weights(df_hotels, beta_saturday)
    v2 = compute_preference_weights(df_hotels, beta_non_saturday)

    S, mix_obj, status_mix = solve_assortment_ortools(
        revenue=revenue,
        v_by_type=[v1, v2],
        theta_by_type=[theta.loc[1], theta.loc[2]],
    )

    S1, _ = optimal_assortment_single_type(df_hotels, v1)
    S2, _ = optimal_assortment_single_type(df_hotels, v2)

    rows.append({
        "dataset": path.name,
        "S_size": int(S.sum()),
        "S1_size": int(S1.sum()),
        "S2_size": int(S2.sum()),
        "mixture_revenue_of_S": mix_obj,
        "revenue_S_under_type1": expected_revenue_type(S, revenue, v1),
        "revenue_S1_under_type1": expected_revenue_type(S1, revenue, v1),
        "type1_gain_from_personalization": expected_revenue_type(S1, revenue, v1) - expected_revenue_type(S, revenue, v1),
        "revenue_S_under_type2": expected_revenue_type(S, revenue, v2),
        "revenue_S2_under_type2": expected_revenue_type(S2, revenue, v2),
        "type2_gain_from_personalization": expected_revenue_type(S2, revenue, v2) - expected_revenue_type(S, revenue, v2),
        "status_mix": interpret_status(status_mix),
        "method_S": "OR-Tools MILP",
        "method_S1": "Revenue-ordered greedy",
        "method_S2": "Revenue-ordered greedy",
    })

problem6_results = pd.DataFrame(rows)
problem6_results

,dataset,S_size,S1_size,S2_size,mixture_revenue_of_S,revenue_S_under_type1,revenue_S1_under_type1,type1_gain_from_personalization,revenue_S_under_type2,revenue_S2_under_type2,type2_gain_from_personalization,status_mix,method_S,method_S1,method_S2
0,data1.csv,18,20,17,107.365996,106.870617,106.877284,0.006667,108.040647,108.041439,0.000793,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
1,data2.csv,10,10,10,130.952105,129.839710,129.839710,0.000000,132.467063,132.467063,0.000000,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
2,data3.csv,18,18,18,121.194251,121.677654,121.677654,0.000000,120.535908,120.535908,0.000000,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy
3,data4.csv,11,11,11,97.511422,96.816135,96.816135,0.000000,98.458326,98.458326,0.000000,OPTIMAL,OR-Tools MILP,Revenue-ordered greedy,Revenue-ordered greedy


## Write-Up Notes

Discuss whether Saturday-night customers have meaningfully different coefficients from non-Saturday customers. Then compare the assortment sizes and revenues against the early-vs-late segmentation from Problem 5.

## Problem 7: AI Agents As Customers

This version uses the Claude front end instead of an API. The notebook exports CSV files for Claude and reads Claude's completed choice CSVs back into the analysis.


In [ ]:
CLAUDE_PART_A_OPTIONS_PATH = Path("problem7_claude_part_a_options.csv")
CLAUDE_PART_A_TEMPLATE_PATH = Path("problem7_claude_part_a_choices_template.csv")
CLAUDE_PART_A_CHOICES_PATH = Path("problem7_claude_part_a_choices.csv")
AI_LABELED_DATA_PATH = Path("problem7_data_ai.csv")

CLAUDE_PREDICT_OPTIONS_PATH = Path("problem7_claude_predict_options.csv")
CLAUDE_PREDICT_EXAMPLES_PATH = Path("problem7_claude_predict_examples.csv")
CLAUDE_PREDICT_TEMPLATE_PATH = Path("problem7_claude_predict_choices_template.csv")
CLAUDE_PREDICT_CHOICES_PATH = Path("problem7_claude_predict_choices.csv")
PREDICT_RESULTS_PATH = Path("problem7_predict_results.csv")
PREDICT_METRICS_PATH = Path("problem7_predict_metrics.csv")

CLAUDE_CHOICE_COLUMNS = ["srch_id", "choice_type", "option_id"]


### Shared Claude CSV Code

In [ ]:
def hotel_options_for_claude(source_df):
    option_columns = [
        "srch_id",
        "row_index",
        "srch_booking_window",
        "srch_adults_count",
        "srch_children_count",
        "srch_room_count",
        "srch_saturday_night_bool",
        *FEATURES,
    ]
    return source_df[option_columns].rename(columns={"row_index": "option_id"})


def choice_template(source_df):
    return pd.DataFrame({
        "srch_id": sorted(source_df["srch_id"].unique()),
        "choice_type": "",
        "option_id": "",
    })


def read_claude_choices(path, valid_options):
    choices = pd.read_csv(path)
    missing_columns = sorted(set(CLAUDE_CHOICE_COLUMNS) - set(choices.columns))
    if missing_columns:
        raise ValueError(f"Claude choices file is missing columns: {missing_columns}")

    choices = choices[CLAUDE_CHOICE_COLUMNS].copy()
    choices["choice_type"] = choices["choice_type"].str.strip().str.lower()
    choices["option_id"] = pd.to_numeric(choices["option_id"], errors="coerce")
    choices["is_valid"] = choices.apply(
        lambda row: (
            row["choice_type"] == "no_purchase" and pd.isna(row["option_id"])
        ) or (
            row["choice_type"] == "hotel"
            and not pd.isna(row["option_id"])
            and int(row["option_id"]) in valid_options.get(row["srch_id"], set())
        ),
        axis=1,
    )
    return choices


### Part A: Simulate Customer Choices With Claude

In [ ]:
problem7_df = pd.read_csv(FULL_DATA).reset_index(names="row_index")
part_a_options = hotel_options_for_claude(problem7_df)
part_a_options.to_csv(CLAUDE_PART_A_OPTIONS_PATH, index=False)
choice_template(problem7_df).to_csv(CLAUDE_PART_A_TEMPLATE_PATH, index=False)
part_a_valid_options = problem7_df.groupby("srch_id")["row_index"].apply(set).to_dict()

print(f"Saved Claude options to {CLAUDE_PART_A_OPTIONS_PATH}")
print(f"Saved Claude response template to {CLAUDE_PART_A_TEMPLATE_PATH}")
print(f"After Claude fills the template, save it as {CLAUDE_PART_A_CHOICES_PATH}")


In [ ]:
ai_choices = read_claude_choices(CLAUDE_PART_A_CHOICES_PATH, part_a_valid_options)
ai_choices.to_csv(CLAUDE_PART_A_CHOICES_PATH, index=False)

assert len(ai_choices) == problem7_df["srch_id"].nunique()
assert not ai_choices["srch_id"].duplicated().any()
assert ai_choices["is_valid"].all(), "Invalid Claude choices exist. Inspect problem7_claude_part_a_choices.csv."

ai_labeled_df = problem7_df.drop(columns=["row_index"]).copy()
ai_labeled_df["booking_bool_ai"] = 0
chosen_row_indices = ai_choices.query("choice_type == 'hotel'")["option_id"].to_numpy(dtype=int)
ai_labeled_df.loc[chosen_row_indices, "booking_bool_ai"] = 1

assert ai_labeled_df.groupby("srch_id")["booking_bool_ai"].sum().le(1).all()
ai_labeled_df.to_csv(AI_LABELED_DATA_PATH, index=False)
print(f"Saved {AI_LABELED_DATA_PATH}")


In [ ]:
X = ai_labeled_df[FEATURES].to_numpy(dtype=float)
groups = ai_labeled_df["srch_id"].to_numpy()

human_result = minimize(
    neg_log_likelihood,
    np.zeros(len(PARAM_NAMES)),
    args=(X, ai_labeled_df["booking_bool"].to_numpy(dtype=float), groups),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)
ai_result = minimize(
    neg_log_likelihood,
    np.zeros(len(PARAM_NAMES)),
    args=(X, ai_labeled_df["booking_bool_ai"].to_numpy(dtype=float), groups),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)

problem7_beta_comparison = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "human_beta": human_result.x,
    "ai_beta": ai_result.x,
})
problem7_beta_comparison["difference_ai_minus_human"] = problem7_beta_comparison["ai_beta"] - problem7_beta_comparison["human_beta"]
problem7_beta_comparison


### Part B: Predict Held-Out Human Choices With Claude

In [ ]:
PREDICT_RANDOM_SEED = 5132
PREDICT_TRAIN_FRAC = 0.80
PREDICT_CONTEXT_EXAMPLES = 5

predict_df = pd.read_csv(FULL_DATA).reset_index(names="row_index")
query_ids = pd.Series(predict_df["srch_id"].unique()).sample(frac=1, random_state=PREDICT_RANDOM_SEED).to_numpy()
train_query_ids = query_ids[: int(PREDICT_TRAIN_FRAC * len(query_ids))]
test_query_ids = query_ids[int(PREDICT_TRAIN_FRAC * len(query_ids)) :]

train_df = predict_df[predict_df["srch_id"].isin(train_query_ids)].copy()
test_df = predict_df[predict_df["srch_id"].isin(test_query_ids)].copy()

mnl_predict_result = minimize(
    neg_log_likelihood,
    np.zeros(len(PARAM_NAMES)),
    args=(
        train_df[FEATURES].to_numpy(dtype=float),
        train_df["booking_bool"].to_numpy(dtype=float),
        train_df["srch_id"].to_numpy(),
    ),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)
beta_predict_mnl = mnl_predict_result.x

print(f"Train queries: {len(train_query_ids):,}")
print(f"Test queries: {len(test_query_ids):,}")


In [ ]:
predict_options = hotel_options_for_claude(test_df)
predict_options.to_csv(CLAUDE_PREDICT_OPTIONS_PATH, index=False)
choice_template(test_df).to_csv(CLAUDE_PREDICT_TEMPLATE_PATH, index=False)
part_b_valid_options = test_df.groupby("srch_id")["row_index"].apply(set).to_dict()

example_rows = []
for srch_id in train_query_ids[:PREDICT_CONTEXT_EXAMPLES]:
    group = train_df[train_df["srch_id"] == srch_id]
    booked_rows = group.loc[group["booking_bool"] == 1, "row_index"]
    example_rows.append({
        "srch_id": int(srch_id),
        "actual_choice_type": "hotel" if len(booked_rows) else "no_purchase",
        "actual_option_id": int(booked_rows.iloc[0]) if len(booked_rows) else None,
    })
pd.DataFrame(example_rows).to_csv(CLAUDE_PREDICT_EXAMPLES_PATH, index=False)

actual_rows = []
for srch_id, group in test_df.groupby("srch_id", sort=True):
    booked_rows = group.loc[group["booking_bool"] == 1, "row_index"]
    actual_rows.append({
        "srch_id": int(srch_id),
        "actual_choice_type": "hotel" if len(booked_rows) else "no_purchase",
        "actual_option_id": int(booked_rows.iloc[0]) if len(booked_rows) else None,
    })
actual_choices = pd.DataFrame(actual_rows)

print(f"Saved held-out Claude options to {CLAUDE_PREDICT_OPTIONS_PATH}")
print(f"Saved Claude examples to {CLAUDE_PREDICT_EXAMPLES_PATH}")
print(f"After Claude fills the template, save it as {CLAUDE_PREDICT_CHOICES_PATH}")


In [ ]:
predict_ai_choices = read_claude_choices(CLAUDE_PREDICT_CHOICES_PATH, part_b_valid_options)
predict_ai_choices = predict_ai_choices.rename(columns={
    "choice_type": "ai_choice_type",
    "option_id": "ai_option_id",
    "is_valid": "ai_is_valid",
})
predict_ai_choices.to_csv(CLAUDE_PREDICT_CHOICES_PATH, index=False)

mnl_predictions = []
for srch_id, group in test_df.groupby("srch_id", sort=True):
    utilities = compute_utilities(beta_predict_mnl, group[FEATURES].to_numpy(dtype=float))
    weights = np.exp(np.clip(utilities, -50, 50))
    best_position = int(np.argmax(weights))
    best_prob = weights[best_position] / (1 + weights.sum())
    no_purchase_prob = 1 / (1 + weights.sum())
    mnl_predictions.append({
        "srch_id": int(srch_id),
        "mnl_choice_type": "hotel" if best_prob > no_purchase_prob else "no_purchase",
        "mnl_option_id": int(group.iloc[best_position]["row_index"]) if best_prob > no_purchase_prob else None,
    })

prediction_results = (
    actual_choices
    .merge(predict_ai_choices, on="srch_id", how="left")
    .merge(pd.DataFrame(mnl_predictions), on="srch_id", how="left")
)

prediction_results["ai_exact_correct"] = (
    prediction_results["ai_is_valid"]
    & (prediction_results["ai_choice_type"] == prediction_results["actual_choice_type"])
    & (prediction_results["ai_option_id"].fillna(-1) == prediction_results["actual_option_id"].fillna(-1))
)
prediction_results["ai_purchase_correct"] = (
    prediction_results["ai_is_valid"]
    & (prediction_results["ai_choice_type"] == prediction_results["actual_choice_type"])
)
prediction_results["mnl_exact_correct"] = (
    (prediction_results["mnl_choice_type"] == prediction_results["actual_choice_type"])
    & (prediction_results["mnl_option_id"].fillna(-1) == prediction_results["actual_option_id"].fillna(-1))
)
prediction_results["mnl_purchase_correct"] = prediction_results["mnl_choice_type"] == prediction_results["actual_choice_type"]
prediction_results.to_csv(PREDICT_RESULTS_PATH, index=False)

prediction_results.head()


In [ ]:
valid_ai = prediction_results[prediction_results["ai_is_valid"]]
problem7_predict_metrics = pd.DataFrame([
    {
        "model": "Claude front end",
        "n_queries": len(valid_ai),
        "exact_choice_accuracy": valid_ai["ai_exact_correct"].mean(),
        "purchase_no_purchase_accuracy": valid_ai["ai_purchase_correct"].mean(),
    },
    {
        "model": "Problem 1 MNL",
        "n_queries": len(prediction_results),
        "exact_choice_accuracy": prediction_results["mnl_exact_correct"].mean(),
        "purchase_no_purchase_accuracy": prediction_results["mnl_purchase_correct"].mean(),
    },
])
problem7_predict_metrics.to_csv(PREDICT_METRICS_PATH, index=False)
problem7_predict_metrics


### Problem 7 Write-Up Notes

For Part A, state that Claude was used through the front end rather than an API. For Part B, report the train/test split, the Claude CSV workflow, exact-choice accuracy, and purchase/no-purchase accuracy.
